# Embedding

In [ ]:
# Pull the outputs from the Preprocessing file

%run ./Preprocessing.ipynb

dict_keys(['dev', 'test', 'train'])
Development Data Frame Columns:  ['source', 'source_labels', 'rouge_scores', 'paper_id', 'target', 'ic', 'title']
Training Data Frame Columns:  ['source', 'source_labels', 'rouge_scores', 'paper_id', 'target', 'ic', 'title']
Testing Data Frame:  ['source', 'source_labels', 'rouge_scores', 'paper_id', 'target', 'ic', 'title']
Total # of Null Values in Development Data Frame:  source           0
source_labels    0
rouge_scores     0
paper_id         0
target           0
ic               0
title            0
dtype: int64
Total # of Null Values in the Training Data Frame:  source           0
source_labels    0
rouge_scores     0
paper_id         0
target           0
ic               0
title            0
dtype: int64
Total # of Null Values in Testing Data Frame:  source           0
source_labels    0
rouge_scores     0
paper_id         0
target           0
ic               0
title            0
dtype: int64
Dataset Name:  dev
Number of Rows in Data Frame: 

In [16]:
# Import necessary libraries

from sentence_transformers import SentenceTransformer
import math

In [8]:
# Load a pretrained embedding model
# Using the BAAI/bge-base-en-v1.5 model since it performs better on scientific text

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

# Try testing one paragraph
tester_text = train_df.loc[0, "source_paragraph"]

# Generate embeddings for the test paragraph
tester_embedding = embedding_model.encode(tester_text, normalize_embeddings = True)

print("Original Tester Text: ", tester_text)
print("Tester Text After Embedding: ", tester_embedding)
print("Shape of Embedded Tester Text: ", tester_embedding.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Original Tester Text:  Due to the success of deep learning to solving a variety of challenging machine learning tasks, there is a rising interest in understanding loss functions for training neural networks from a theoretical aspect. Particularly, the properties of critical points and the landscape around them are of importance to determine the convergence performance of optimization algorithms. In this paper, we provide a necessary and sufficient characterization of the analytical forms for the critical points (as well as global minimizers) of the square loss functions for linear neural networks. We show that the analytical forms of the critical points characterize the values of the corresponding loss functions as well as the necessary and sufficient conditions to achieve global minimum. Furthermore, we exploit the analytical forms of the critical points to characterize the landscape properties for the loss functions of linear neural networks and shallow ReLU networks. One particular 

In [22]:
# Figure out what the average length is for sources values
print("Average Length of Source Parargraphs: ", train_df["source_paragraph"].apply(lambda x: len(x.split())).mean())

# Create a function to chunk the paragraphs
# Since the average length of the paragraphs is about 1000, I'll move forward with a chunksize of 300.
# Add a 50-word overlap so ideas are not split

############################################################
                    # CHUNKING PARAMETERS 
############################################################
set_chunk_size = 300
set_overlap = 50
set_min_chunk_size = 50

def paragraph_chunks(paragraph, chunk_size = set_chunk_size, overlap = set_overlap, min_chunk_size = set_min_chunk_size):

    # First split up the paragraphs into individual words
    words = paragraph.split()

    # Create an empty list to store the chunks
    chunks = []

    # Calculate the step size by subtracting the overlap from the chunk size
    step = chunk_size - overlap

    # Create the overlapping chunks
    for start in range(0, len(words), step):
        
        # Calculate the end
        end = start + chunk_size 

        # Join the words inton chunks
        chunk = " ".join(words[start:end])

        # Append the chunks after stripping
        if chunk.strip():
            chunks.append(chunk)

    return chunks

# Test the paragraph chunking function on a sample paragraph
sample_chunks = paragraph_chunks(train_df.loc[0, "source_paragraph"])

# Apply the function to the tester
sample_chunks = paragraph_chunks(tester_text)

# Test to see if the chunking function properly worked
tester_length = len(tester_text.split())
expected_chunks = math.ceil((tester_length - 300) / 250) + 1

print("Length of Tester Text: ", tester_length)
print("# of Chunks from Tester Text Expected: ", expected_chunks)
print("# of Chunks from Tester Text: ", len(sample_chunks))

Average Length of Source Parargraphs:  995.1902610441767
Length of Tester Text:  1119
# of Chunks from Tester Text Expected:  5
# of Chunks from Tester Text:  5


In [ ]:
# Apply the chunking mechanism to each paragraph
train_df["source_chunks"] = train_df["source_paragraph"].apply(paragraph_chunks)

# Now create a separate data frame to store the chunks
train_df_chunked = train_df.explode("source_chunks").copy()

# Check the chunks are properly copied
# print(train_df['source_paragraph'][0])
# print(train_df_chunked['source_chunks'].head(5))
print(train_df_chunked)

                                                 source  \
0     [Due to the success of deep learning to solvin...   
0     [Due to the success of deep learning to solvin...   
0     [Due to the success of deep learning to solvin...   
0     [Due to the success of deep learning to solvin...   
0     [Due to the success of deep learning to solvin...   
...                                                 ...   
1990  [Machine learned large-scale retrieval systems...   
1991  [The ability to autonomously explore and navig...   
1991  [The ability to autonomously explore and navig...   
1991  [The ability to autonomously explore and navig...   
1991  [The ability to autonomously explore and navig...   

                                          source_labels  \
0     [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
0     [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
0     [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
0     [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...

In [37]:
# Reset the index for the new chunked training df
train_df_chunked = train_df_chunked.reset_index(drop = True)

# Rank the order of the chunk within its respective paragraph
train_df_chunked['chunk_no'] = train_df_chunked.groupby("paper_id").cumcount() + 1

# Double check that the chunks were numbered correctly
print(train_df_chunked[['source_chunks', 'chunk_no']])

                                          source_chunks  chunk_no
0     Due to the success of deep learning to solving...         1
1     aspects such as generalization error, represen...         2
2     of matrix factorization problems and deep line...         3
3     these landscape properties.2) For the square l...         4
4     namely, shallow linear networks, deep linear n...         5
...                                                 ...       ...
8915  proposed model architecture is slightly differ...         4
8916  The ability to autonomously explore and naviga...         1
8917  scene geometry, producing an explicit geometri...         2
8918  "as needed". Specifically, we employ off-the-s...         3
8919  between human and autonomous performance, indi...         4

[8920 rows x 2 columns]
